# Multimodal Semantic Search — Interactive Demo (Google Colab)

Launches the Gradio demo against the trained model. Produces a **public URL** (`https://*.gradio.live`) that anyone can visit during the presentation — works without anyone needing to install anything locally.

**Prerequisites** — these should already be on Drive from the training run:

```
MyDrive/dl-multimodal/
├── cached/                          ← 6 cached feature/json files
├── flickr30k-images/                ← real Flickr30k images (31k files)
├── results.csv                      ← captions
└── output/
    ├── checkpoint_epoch20_learnable.pt
    ├── gallery_embs.pt              ← 31k pre-encoded gallery
    └── gallery_ids.json
```

If `output/` is missing, run [03_training.ipynb](03_training.ipynb) first.

**Runtime → Change runtime type → GPU (T4)** — not strictly required but makes text encoding faster.

## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

## 2. Verify Drive layout

In [ ]:
DRIVE = '/content/drive/MyDrive/dl-multimodal'
!ls {DRIVE}
!ls {DRIVE}/output

## 3. Clone repo and install dependencies

In [ ]:
%cd /content
!rm -rf dl-multimodal-semantic-search
!git clone https://github.com/nlklfor/dl-multimodal-semantic-search.git
%cd dl-multimodal-semantic-search
!git checkout dev

In [ ]:
# Colab pre-installs torch / transformers; gradio is the only one missing.
!pip install -q transformers gradio

## 4. Wire Drive artefacts into the project's expected paths

In [ ]:
import os, json

DRIVE = '/content/drive/MyDrive/dl-multimodal'
os.makedirs('data/cached',              exist_ok=True)
os.makedirs('data/raw',                 exist_ok=True)
os.makedirs('experiments/checkpoints',  exist_ok=True)

# Cached ResNet-50 features (only needed if --split is used; harmless to link).
for f in os.listdir(f'{DRIVE}/cached'):
    src, dst = f'{DRIVE}/cached/{f}', f'data/cached/{f}'
    if not os.path.exists(dst): os.symlink(src, dst)

# Pre-computed 31k gallery + ids.
for f in ['gallery_embs.pt', 'gallery_ids.json']:
    src = f'{DRIVE}/output/{f}'
    dst = f'data/cached/{f}'
    if os.path.exists(src) and not os.path.exists(dst):
        os.symlink(src, dst)

# Trained checkpoint.
ckpt_src = f'{DRIVE}/output/checkpoint_epoch50_learnable.pt'
ckpt_dst = 'experiments/checkpoints/checkpoint_epoch50_learnable.pt'
if not os.path.exists(ckpt_dst):
    os.symlink(ckpt_src, ckpt_dst)

# Real Flickr30k images (~31k files). NOTE: do not os.listdir() this folder —
# Drive's FUSE driver throttles bulk listings of large directories. The demo
# reads images by exact filename via Image.open(), which works fine.
if not os.path.exists('data/raw/flickr30k-images'):
    os.symlink(f'{DRIVE}/flickr30k-images', 'data/raw/flickr30k-images')

# Lightweight sanity check: pick 3 random IDs from gallery_ids and verify
# those specific files exist. Bypasses the listdir() problem.
print('Wired symlinks:')
!ls -l data/cached/gallery_embs.pt data/cached/gallery_ids.json
!ls -l experiments/checkpoints/checkpoint_epoch50_learnable.pt

ids = json.load(open('data/cached/gallery_ids.json'))
print(f'\nGallery has {len(ids):,} image IDs. Spot-checking 3 of them on disk:')
for i in (0, len(ids) // 2, len(ids) - 1):
    path = f'data/raw/flickr30k-images/{ids[i]}'
    ok   = os.path.exists(path)
    size = os.path.getsize(path) if ok else 0
    flag = '✓' if ok and size > 1000 else '✗'
    print(f'  {flag}  {ids[i]}  ({size:,} bytes)')

## 5. Quick sanity check before launching

In [ ]:
import torch, json
from src.encoders.text_encoder   import TextEncoder
from src.encoders.vision_encoder import VisionEncoder

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

vision = VisionEncoder().to(device)
text   = TextEncoder().to(device)
ckpt = torch.load('experiments/checkpoints/checkpoint_epoch50_learnable.pt',
                  map_location=device, weights_only=False)
vision.load_state_dict(ckpt['vision_encoder'])
text.load_state_dict(ckpt['text_encoder'])
print(f'Loaded checkpoint from epoch {ckpt["epoch"]}, val loss {ckpt["loss"]:.4f}')

gallery = torch.load('data/cached/gallery_embs.pt', weights_only=True)
ids     = json.load(open('data/cached/gallery_ids.json'))
print(f'Gallery: {gallery.shape[0]:,} images × {gallery.shape[1]}-d')

# Smoke search: encode a query and verify the top-5 image filenames are reasonable
text.eval()
with torch.no_grad():
    q = text(['a dog running on the beach'], device).cpu()
top5 = (q @ gallery.T).squeeze(0).topk(5).indices.tolist()
print('Top-5 for "a dog running on the beach":')
for idx in top5:
    print(f'  {idx:>5}  {ids[idx]}')

## 6. Launch the Gradio demo

The `--share` flag creates a public URL like `https://*.gradio.live`. The cell will keep running until you stop it (the URL stays alive ~72 h).

**Demo queries to try** (matches the spec from #14):
- `"a dog running on the beach"`
- `"two people sharing an umbrella"`
- `"a child blowing out birthday candles"`
- `"cyclists racing on a mountain road"`

In [ ]:
!python src/demo/app.py \
    --checkpoint experiments/checkpoints/checkpoint_epoch50_learnable.pt \
    --share

## Optional — run a few canned queries from Python instead of the UI

Useful if you want to grab screenshots for the report without recording the live UI. Encodes the query, ranks the gallery, and displays the top-5 images inline in the notebook.

In [ ]:
import torch, json, matplotlib.pyplot as plt
from PIL import Image

queries = [
    'a dog playing in the snow',
    'two people sharing an umbrella',
    'a child blowing out birthday candles',
    'cyclists racing on a mountain road',
]

for q in queries:
    with torch.no_grad():
        q_emb = text([q], device).cpu()
    top5 = (q_emb @ gallery.T).squeeze(0).topk(5).indices.tolist()

    fig, axes = plt.subplots(1, 5, figsize=(15, 3))
    fig.suptitle(f'Query: {q!r}', fontsize=12)
    for ax, idx in zip(axes, top5):
        img = Image.open(f'data/raw/flickr30k-images/{ids[idx]}').convert('RGB')
        ax.imshow(img); ax.axis('off')
    plt.tight_layout(); plt.show()